In [30]:
import os
import json
from pprint import pprint
from typing import Any, Dict, List

In [31]:
taint_result_root_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone"

In [32]:
# Function get all sub-directory under root directory
def get_all_subdirectories_under_one_level(root_dir: str):
    """
    Get all subdirectories directly under the given root directory (one level only).
    Return a list of full paths.
    """
    return [
        os.path.join(root_dir, name)
        if not os.path.isabs(name) else name
        for name in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, name))
    ]
# Function read JSON file (results.json) if it exist
def read_json_if_exists(file_path: str) -> dict:
    """
    Read a JSON file and return its content as a dictionary.
    If the file does not exist or is not a valid JSON, return an empty dict.
    """
    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data
    except json.JSONDecodeError as e:
        print(f"JSON decode error in {file_path}: {e}")
        return {}
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return {}
def get_app_package_name(path: str) -> str:
    """
    Return the last directory name from a full path.
    Example:
        E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote
        -> 'adarshurs.android.vlcmobileremote'
    """
    return os.path.basename(os.path.normpath(path))
def extract_sink_source_from_compliance(compliance_info: Dict[str, Any]) -> List[dict]:
    """
    compliance_info: dict tại khóa 'ComplianceInfo' trong results.json

    Trả về: danh sách dict gồm
      - category  : ví dụ 'PersonalInfo', 'Location', ...
      - rule_name : ví dụ 'PersonalInfo_Network', 'Location_NetworkExternal', ...
      - sink      : nội dung details['Sink'] (có thể là str/dict/list tùy dữ liệu)
      - source    : nội dung details['Source'] (có thể là str/dict/list tùy dữ liệu)
    """
    results: List[dict] = []

    if not isinstance(compliance_info, dict):
        return results

    # compliance_info có dạng: { "PersonalInfo": {...}, "Location": {...}, ... }
    for category_name, category_obj in compliance_info.items():
        if not isinstance(category_obj, dict):
            continue

        # Mỗi category chứa các rule: { "PersonalInfo_Network": { ... }, ... }
        for rule_name, rule_obj in category_obj.items():
            if not isinstance(rule_obj, dict):
                continue

            vulners = rule_obj.get("vulners", [])
            if not isinstance(vulners, list):
                continue

            for vuln in vulners:
                if not isinstance(vuln, dict):
                    continue

                details = vuln.get("details", {})
                if not isinstance(details, dict):
                    continue

                sink = details.get("Sink", None)
                source = details.get("Source", None)

                results.append({
                    "category": category_name,
                    "rule_name": rule_name,
                    "sink": sink,
                    "source": source,
                })

    return results
def create_directory_if_not_exist(path: str):
    """Tạo thư mục nếu chưa tồn tại."""
    if not os.path.exists(path):
        os.makedirs(path)
def write_result_to_txt(result_dict, filename, file_path="."):
    """
    Ghi nội dung của dict kết quả vào file .txt trong thư mục chỉ định.

    Parameters:
        result_dict (dict): dict chứa kết quả cần ghi
        filename (str): tên file (ví dụ: 'output.txt' hoặc 'result')
        file_path (str): đường dẫn thư mục để lưu file (mặc định là thư mục hiện tại)
    """
    try:
        # Đảm bảo thư mục tồn tại
        os.makedirs(file_path, exist_ok=True)

        # Đảm bảo file có đuôi .txt
        if not filename.endswith(".txt"):
            filename += ".txt"

        # Tạo full path
        full_path = os.path.join(file_path, filename)

        # Ghi từng cặp key-value ra file
        with open(full_path, "w", encoding="utf-8") as f:
            for key, value in result_dict.items():
                f.write(f"{key}: {value}\n")

        # print(f"Write to file: {full_path}")

    except Exception as e:
        print(f"Error: {e}")

In [33]:
level_1_directory_arr = get_all_subdirectories_under_one_level(taint_result_root_directory)
# print(package_name_arr)

In [34]:
output_root_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-sharing-non-compliance-evaluation\source_sink"

In [35]:
for i, level_1_path in enumerate(level_1_directory_arr):
    print("Level_1 path:", level_1_path)
    package_name = get_app_package_name(level_1_path)
    print(f"==== Package {i}: {package_name} ====")
    output_source_sink_path = output_root_directory+"\\"+package_name
    create_directory_if_not_exist(output_source_sink_path)
    
    taint_result_dict = read_json_if_exists(os.path.join(level_1_path, "results.json"))
    compliance_info = taint_result_dict.get("ComplianceInfo", {})

    rows = extract_sink_source_from_compliance(compliance_info)

    # ví dụ in nhanh
    for idx, r in enumerate(rows):  # index bắt đầu từ 0
        result = {
            "app_package_name": package_name,
            "category": r["category"],
            "rule_name": r["rule_name"],
            "Sink": r["sink"],
            "Source": r["source"],
        }
        file_name = f"{package_name}-source-sink-{idx}"
        write_result_to_txt(result, file_name,output_source_sink_path)
        # break
    # break

Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\app.groupcal.www.apk
==== Package 0: app.groupcal.www.apk ====
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\apps.r.compass.apk
==== Package 1: apps.r.compass.apk ====
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\ch.publisheria.bring.apk
==== Package 2: ch.publisheria.bring.apk ====
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\com.anghami.apk
==== Package 3: com.anghami.apk ====
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\com.audible.application.apk
==== Package 4: com.audible.application.apk ====
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\com.c25k.apk
==== Package 5: com.c25k.apk ====
Level_1 path: C:\U